In [3]:
!pip install playwright nest-asyncio pandas
!playwright install chromium

(node:86422) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
159.6 MiB [                    ] 0% 0.0s159.6 MiB [                    ] 0% 320.4s159.6 MiB [                    ] 0% 903.1s159.6 MiB [                    ] 0% 737.9s159.6 MiB [                    ] 0% 621.0s159.6 MiB [                    ] 0% 832.0s159.6 MiB [                    ] 0% 690.7s159.6 MiB [                    ] 0% 805.5s159.6 MiB [                    ] 0% 713.8s159.6 MiB [                    ] 0% 645.4s159.6 MiB [                    ] 0% 593.5s159.6 MiB [                    ] 0% 562.3s159.6 MiB [                    ] 0% 576.7s159.6 MiB [                    ] 0% 540.4s159.6 MiB [                    ] 0% 514.3s159.6 MiB [                    ] 0% 486.0s159.6 MiB [                  

In [ ]:
import asyncio
import nest_asyncio
import pandas as pd
import random
import time
import os
import re
import sys
from datetime import datetime, timedelta
from playwright.async_api import async_playwright

nest_asyncio.apply()

# ============================================================
# 📁 CONFIG
# ============================================================
INPUT_FILE = "./raw.csv"               # CSV: hotel_name, hotel_url, room_type
TEMP_OUTPUT_FILE = "hotel_prices_temp.csv"
OUTPUT_PREFIX = "hotel_prices_"

# ⚙️ PERFORMANCE
NUM_WORKERS = 4          # Số hotels crawl song song
WEEKS_PER_HOTEL = 3      # Số weeks crawl song song / hotel
MAX_RETRIES = 2          # Số lần retry mỗi week (trong 1 lần crawl)
DELAY_RANGE = (0.5, 1.5) # Delay giữa requests
HOTEL_DELAY = (1, 2)     # Delay giữa mỗi hotel
PAGE_TIMEOUT = 15000     # Page load timeout (ms)

# 🔁 BATCH RETRY CONFIG
BATCH_SIZE = 10           # Số hotels mỗi batch
MAX_RETRY_ROUNDS = 2      # Số vòng retry NA sau mỗi batch
TARGET_NA_RATE = 0.10     # Dừng retry khi NA ≤ 10%
RETRY_COOL_DOWN = (3, 6) # Nghỉ giữa các vòng retry (giây)

# 🔁 RETRY ESCALATION - tăng dần qua mỗi round
RETRY_PAGE_TIMEOUT = [20000, 30000]   # Timeout tăng dần
RETRY_WAIT_STRATEGY = ["domcontentloaded", "networkidle"]

# 🎭 STEALTH
USER_AGENTS = [
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:122.0) Gecko/20100101 Firefox/122.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
]

STEALTH_SCRIPT = """
Object.defineProperty(navigator, 'webdriver', {get: () => undefined});
Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3, 4, 5]});
Object.defineProperty(navigator, 'languages', {get: () => ['en-US', 'en']});
window.chrome = { runtime: {} };
const originalQuery = window.navigator.permissions.query;
window.navigator.permissions.query = (parameters) => (
    parameters.name === 'notifications'
    ? Promise.resolve({state: Notification.permission})
    : originalQuery(parameters)
);
"""

# ============================================================
# 🔍 JS EXTRACTION - SOLD OUT + DUAL PRICE FORMAT (₫ prefix & suffix)
# ============================================================
EXTRACT_PRICES_JS = """(targetRoom) => {
    const results = [];
    const masterRooms = document.querySelectorAll("[data-selenium='MasterRoom']");
    const targetLower = targetRoom.toLowerCase().trim();

    // ── Helper: extract price from text (both ₫1,234 and 1.234₫ formats) ──
    function findPriceInText(text) {
        // Format 1: ₫ 1,507,222 or ₫1,507,222
        const m1 = text.match(/₫\\s*[\\d,.\\ ]+/);
        if (m1) return m1[0].trim();
        // Format 2: 5.193.000₫ or 5,193,000₫ or 5.193.000 ₫
        const m2 = text.match(/[\\d,.]+\\s*₫/);
        if (m2) return m2[0].trim();
        // Format 3: VND prefix/suffix
        const m3 = text.match(/VND\\s*[\\d,.\\ ]+|[\\d,.]+\\s*VND/i);
        if (m3) return m3[0].trim();
        return null;
    }

    // ── Check 1: Hotel-level sold out ──
    const bodyText = document.body.innerText || '';
    const hotelSoldOut = /sold\\s*out[!.]?\\s*(our last room|all rooms)/i.test(bodyText);
    // Also check for "no rooms available" / "no availability"
    const noAvailability = /no\\s*(rooms?)?\\s*avail/i.test(bodyText) && masterRooms.length === 0;

    if ((hotelSoldOut || noAvailability) && masterRooms.length === 0) {
        return {found: false, soldOut: true, soldOutType: 'hotel', allRooms: 0};
    }

    masterRooms.forEach(room => {
        const nameEl = room.querySelector("[data-selenium='masterroom-title-name']");
        const name = nameEl ? nameEl.textContent.trim() : '';
        const roomText = room.innerText || '';

        // ── Check 2: Room-level sold out ──
        let soldOutPrice = null;
        // "Sold out at ₫ 971,393" or "Sold out at 971.393₫"
        const soldOutMatch = roomText.match(/sold\\s*out\\s*at\\s*([₫đ]\\s*[\\d,.\\ ]+|[\\d,.]+\\s*[₫đ])/i);
        if (soldOutMatch) {
            soldOutPrice = soldOutMatch[1].trim();
        }
        if (!soldOutPrice) {
            let parent = room.parentElement;
            for (let i = 0; i < 3 && parent; i++) {
                const parentText = parent.innerText || '';
                const parentMatch = parentText.match(/sold\\s*out\\s*at\\s*([₫đ]\\s*[\\d,.\\ ]+|[\\d,.]+\\s*[₫đ])/i);
                if (parentMatch) {
                    soldOutPrice = parentMatch[1].trim();
                    break;
                }
                parent = parent.parentElement;
            }
        }

        // ── Normal price extraction ──
        let prices = [];

        // Strategy 1: PriceDisplay elements
        room.querySelectorAll("[data-selenium='PriceDisplay']").forEach(p => {
            const text = p.textContent.trim();
            if (text) prices.push(text);
        });

        // Strategy 2: Traverse UP
        if (prices.length === 0) {
            let parent = room.parentElement;
            for (let i = 0; i < 3 && parent; i++) {
                parent.querySelectorAll("[data-selenium='PriceDisplay']").forEach(p => {
                    const text = p.textContent.trim();
                    if (text) prices.push(text);
                });
                if (prices.length > 0) break;
                parent = parent.parentElement;
            }
        }

        // Strategy 3: Find price-like text (BOTH formats: ₫X and X₫)
        if (prices.length === 0) {
            const walker = document.createTreeWalker(room, NodeFilter.SHOW_TEXT);
            while (walker.nextNode()) {
                const text = walker.currentNode.textContent.trim();
                const price = findPriceInText(text);
                if (price) prices.push(price);
            }
        }

        results.push({
            name: name,
            nameLower: name.toLowerCase().trim(),
            prices: prices.slice(0, 5),
            matched: name.toLowerCase().trim() === targetLower,
            soldOutPrice: soldOutPrice
        });
    });

    // ── Find target room ──
    const target = results.find(r => r.matched);
    if (target) {
        if (target.prices.length > 0) {
            return {found: true, price: target.prices[0], room: target.name, allRooms: results.length};
        }
        if (target.soldOutPrice) {
            return {found: false, soldOut: true, soldOutType: 'room', soldOutPrice: target.soldOutPrice, room: target.name, allRooms: results.length};
        }
    }

    // ── Fallback: partial match ──
    const partial = results.find(r =>
        r.nameLower.includes(targetLower) || targetLower.includes(r.nameLower)
    );
    if (partial) {
        if (partial.prices.length > 0) {
            return {found: true, price: partial.prices[0], room: partial.name, allRooms: results.length, partial: true};
        }
        if (partial.soldOutPrice) {
            return {found: false, soldOut: true, soldOutType: 'room', soldOutPrice: partial.soldOutPrice, room: partial.name, allRooms: results.length, partial: true};
        }
    }

    // ── All rooms sold out ──
    const allSoldOut = results.length > 0 && results.every(r => r.soldOutPrice && r.prices.length === 0);
    if (allSoldOut) {
        const relevantRoom = target || partial || results[0];
        return {found: false, soldOut: true, soldOutType: 'all_rooms', soldOutPrice: relevantRoom.soldOutPrice, allRooms: results.length};
    }

    return {found: false, soldOut: false, allRooms: results.length, roomNames: results.map(r => r.name)};
}"""

# ============================================================
# 📁 CSV READ / SAVE
# ============================================================
def read_hotels_from_csv(file_path):
    try:
        df = pd.read_csv(file_path)
        required_cols = ['hotel_name', 'hotel_url', 'room_type']
        if not all(col in df.columns for col in required_cols):
            print(f"⚠️  CSV columns không chuẩn. Dùng 3 columns đầu.", flush=True)
            df.columns = ['hotel_name', 'hotel_url', 'room_type'] + list(df.columns[3:])
        df = df[df['hotel_url'].notna() & (df['hotel_url'] != '')]
        print(f"✅ Đọc được {len(df)} hotels từ {file_path}", flush=True)
        return df[['hotel_name', 'hotel_url', 'room_type']]
    except Exception as e:
        print(f"❌ Lỗi đọc CSV: {e}", flush=True)
        return pd.DataFrame(columns=['hotel_name', 'hotel_url', 'room_type'])

def save_backup_csv(all_week_prices, filename):
    try:
        rows = []
        for (hotel, room), prices in all_week_prices.items():
            row = {"hotel_name": hotel, "room_type": room}
            for i in range(1, 7):
                row[f"price_w{i}"] = prices.get(f"Price W{i}", "NA")
            rows.append(row)
        df = pd.DataFrame(rows)
        df.to_csv(filename, index=False)
    except Exception as e:
        print(f"❌ Error saving: {e}", flush=True)

def update_url_checkin(url, checkin_date):
    new_date = checkin_date.strftime("%Y-%m-%d")
    if 'checkin=' in url.lower():
        url = re.sub(r'checkin=[\d-]+', f'checkin={new_date}', url, flags=re.IGNORECASE)
    else:
        separator = "&" if "?" in url else "?"
        url = f"{url}{separator}checkin={new_date}"
    return url

# ============================================================
# 📊 NA RATE HELPERS - SOLD OUT is NOT counted as NA
# ============================================================
def calc_na_stats(prices_dict):
    total = 6
    na_count = sum(1 for i in range(1, 7) if prices_dict.get(f"Price W{i}", "NA") == "NA")
    return na_count, total

def calc_batch_na_rate(all_week_prices, batch_keys):
    total_cells = 0
    na_cells = 0
    for key in batch_keys:
        if key in all_week_prices:
            na, t = calc_na_stats(all_week_prices[key])
            na_cells += na
            total_cells += t
    return na_cells / max(total_cells, 1), na_cells, total_cells

def find_na_weeks(all_week_prices, batch_keys):
    na_items = []
    for key in batch_keys:
        if key not in all_week_prices:
            continue
        prices = all_week_prices[key]
        for i in range(1, 7):
            val = prices.get(f"Price W{i}", "NA")
            if val == "NA":
                na_items.append((key, i))
    return na_items


# ============================================================
# 🎭 BROWSER MANAGER - Auto-relaunch on crash
# ============================================================
class BrowserManager:
    def __init__(self, playwright):
        self._playwright = playwright
        self._browser = None

    async def get_browser(self):
        if self._browser and self._browser.is_connected():
            return self._browser
        if self._browser:
            print("🔄 Browser crashed, relaunching...", flush=True)
            try:
                await self._browser.close()
            except:
                pass
        self._browser = await self._playwright.chromium.launch(
            headless=True,
            args=[
                '--disable-blink-features=AutomationControlled',
                '--no-sandbox',
                '--disable-dev-shm-usage',
            ]
        )
        print("✅ Browser launched", flush=True)
        return self._browser

    async def close(self):
        if self._browser:
            try:
                await self._browser.close()
            except:
                pass

# ============================================================
# 🎭 CRAWL 1 WEEK - WITH SOLD OUT + ESCALATION SUPPORT
# ============================================================
async def crawl_single_week(browser_mgr, hotel_url, room_type, week_num, checkin,
                            retries=None, page_timeout=None, wait_until=None):
    if retries is None:
        retries = MAX_RETRIES
    if page_timeout is None:
        page_timeout = PAGE_TIMEOUT
    if wait_until is None:
        wait_until = "domcontentloaded"

    result = {"week": week_num, "price": "NA", "date": checkin.strftime('%Y-%m-%d')}

    for retry in range(retries):
        context = None
        try:
            if retry > 0:
                backoff = random.uniform(2, 4)
                print(f"      🔄 W{week_num} retry {retry}/{retries} ({backoff:.1f}s)", flush=True)
                await asyncio.sleep(backoff)

            browser = await browser_mgr.get_browser()
            context = await browser.new_context(
                viewport={"width": random.randint(1366, 1920), "height": random.randint(768, 1080)},
                user_agent=random.choice(USER_AGENTS),
                locale="en-US",
            )
            page = await context.new_page()
            await page.add_init_script(STEALTH_SCRIPT)

            url = update_url_checkin(hotel_url, checkin)

            # Dùng wait_until theo strategy (domcontentloaded hoặc networkidle)
            try:
                await page.goto(url, timeout=page_timeout, wait_until=wait_until)
            except Exception:
                # Fallback: nếu networkidle timeout thì thử domcontentloaded
                if wait_until == "networkidle":
                    try:
                        await page.goto(url, timeout=page_timeout, wait_until="domcontentloaded")
                    except:
                        pass
                else:
                    raise

            await asyncio.sleep(random.uniform(0.5, 1.0))

            # Human-like scroll
            await page.evaluate("window.scrollTo({top: 300, behavior: 'smooth'})")
            await asyncio.sleep(random.uniform(0.5, 1.0))

            # Close popup
            try:
                close_btn = page.locator(".ab-close-button")
                if await close_btn.count() > 0:
                    await close_btn.first.click(timeout=2000)
            except:
                pass

            # Wait for price elements
            try:
                await page.wait_for_selector("[data-selenium='PriceDisplay']", timeout=5000)
            except:
                try:
                    await page.wait_for_selector("div#roomGrid", timeout=5000)
                    await asyncio.sleep(5)
                except:
                    # Extra wait khi dùng escalation strategy
                    extra_wait = 2
                    await asyncio.sleep(extra_wait)

            # Scroll to trigger lazy loading (v2 pattern - proven stable)
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight / 2)")
            await asyncio.sleep(0.5)
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            await asyncio.sleep(random.uniform(0.5, 1.0))

            extraction = await page.evaluate(EXTRACT_PRICES_JS, room_type)

            # Case 1: Found price
            if extraction.get('found'):
                price = extraction['price']
                if extraction.get('partial'):
                    matched_name = extraction.get('room', '')
                    print(f"      ✅ W{week_num}: {price} | {checkin.strftime('%Y-%m-%d')} (partial: '{matched_name}')", flush=True)
                else:
                    print(f"      ✅ W{week_num}: {price} | {checkin.strftime('%Y-%m-%d')}", flush=True)
                result["price"] = price
                return result

            # Case 2: Sold out → ghi nhận, KHÔNG retry
            if extraction.get('soldOut'):
                sold_type = extraction.get('soldOutType', 'unknown')
                sold_price = extraction.get('soldOutPrice', '')
                if sold_type == 'hotel':
                    print(f"      🚫 W{week_num}: SOLD OUT (toàn bộ hotel) | {checkin.strftime('%Y-%m-%d')}", flush=True)
                    result["price"] = "SOLD OUT"
                elif sold_type == 'all_rooms':
                    print(f"      🚫 W{week_num}: SOLD OUT (all rooms) {sold_price} | {checkin.strftime('%Y-%m-%d')}", flush=True)
                    result["price"] = f"SOLD OUT {sold_price}"
                else:
                    room_name = extraction.get('room', room_type)
                    print(f"      🚫 W{week_num}: SOLD OUT {sold_price} ({room_name}) | {checkin.strftime('%Y-%m-%d')}", flush=True)
                    result["price"] = f"SOLD OUT {sold_price}"
                return result

            # Case 3: Not found → retry
            else:
                if retry == retries - 1:
                    rooms_on_page = extraction.get('allRooms', 0)
                    if rooms_on_page == 0:
                        print(f"      ❌ W{week_num}: NA (no rooms loaded)", flush=True)
                    else:
                        room_names = extraction.get('roomNames', [])
                        print(f"      ❌ W{week_num}: NA ({rooms_on_page} rooms: {room_names[:3]})", flush=True)

        except Exception as e:
            err_msg = str(e)
            is_browser_crash = "has been closed" in err_msg or "Target page" in err_msg
            if is_browser_crash:
                print(f"      ⚠️ W{week_num}: Browser crashed, will relaunch on next retry", flush=True)
            elif retry == retries - 1:
                print(f"      ❌ W{week_num}: NA ({err_msg[:60]})", flush=True)
        finally:
            if context:
                try:
                    await context.close()
                except:
                    pass

        await asyncio.sleep(random.uniform(*DELAY_RANGE))

    return result

# ============================================================
# 🏨 PROCESS 1 HOTEL (initial crawl)
# ============================================================
async def process_hotel(browser_mgr, hotel_info, prev_data, base_checkin, week_offsets, hotel_semaphore):
    hotel_name, hotel_url, room_type = hotel_info
    key = (hotel_name, room_type)

    if key in prev_data:
        if all(prev_data[key].get(f"Price W{i}", "NA") != "NA" for i in range(1, 7)):
            print(f"⏭️  SKIP: {hotel_name}", flush=True)
            return key, prev_data[key], True

    async with hotel_semaphore:
        await asyncio.sleep(random.uniform(*HOTEL_DELAY))
        print(f"\n🏨 START: {hotel_name} | Room: {room_type}", flush=True)

        prices = {}
        weeks_to_crawl = []

        for week_num, offset in enumerate(week_offsets, start=1):
            key_prefix = f"Price W{week_num}"
            if key in prev_data and prev_data[key].get(key_prefix, "NA") != "NA":
                prices[key_prefix] = prev_data[key][key_prefix]
                print(f"   ✓ W{week_num}: {prices[key_prefix]} (cached)", flush=True)
            else:
                weeks_to_crawl.append((week_num, offset))

        if weeks_to_crawl:
            print(f"   🚀 Crawling {len(weeks_to_crawl)} weeks (max {WEEKS_PER_HOTEL} parallel)...", flush=True)

            week_semaphore = asyncio.Semaphore(WEEKS_PER_HOTEL)

            async def crawl_with_limit(wn, offset):
                async with week_semaphore:
                    checkin = base_checkin + timedelta(days=offset)
                    return await crawl_single_week(browser_mgr, hotel_url, room_type, wn, checkin)

            tasks = [crawl_with_limit(wn, off) for wn, off in weeks_to_crawl]
            results = await asyncio.gather(*tasks)

            for r in results:
                prices[f"Price W{r['week']}"] = r["price"]

        na_count, _ = calc_na_stats(prices)
        sold_count = sum(1 for i in range(1, 7) if str(prices.get(f"Price W{i}", "")).startswith("SOLD OUT"))
        if na_count == 0 and sold_count == 0:
            status_icon = "✅"
        elif na_count == 0 and sold_count > 0:
            status_icon = f"🚫 ({sold_count} sold out)"
        else:
            status_icon = f"⚠️ ({na_count} NA, {sold_count} sold out)"
        print(f"{status_icon} DONE: {hotel_name}", flush=True)
        return key, prices, False

# ============================================================
# 🔁 RETRY NA - ESCALATION STRATEGY
# ============================================================
async def retry_na_for_batch(browser_mgr, batch_infos, batch_keys, all_week_prices, base_checkin, week_offsets):
    """
    Retry NA với escalation:
    - Round 1: domcontentloaded + timeout 25s (giống initial)
    - Round 2: networkidle + timeout 35s (chờ AJAX xong)
    - Round 3: networkidle + timeout 45s (max patience)
    """
    key_to_info = {}
    for info in batch_infos:
        key = (info[0], info[2])
        key_to_info[key] = info

    for round_num in range(1, MAX_RETRY_ROUNDS + 1):
        na_rate, na_cells, total_cells = calc_batch_na_rate(all_week_prices, batch_keys)

        if na_rate <= TARGET_NA_RATE:
            print(f"   🎯 Batch NA rate: {na_rate:.1%} ({na_cells}/{total_cells}) ≤ {TARGET_NA_RATE:.0%} → OK!", flush=True)
            return

        na_items = find_na_weeks(all_week_prices, batch_keys)
        if not na_items:
            return

        # Escalation params
        round_idx = min(round_num - 1, len(RETRY_PAGE_TIMEOUT) - 1)
        timeout = RETRY_PAGE_TIMEOUT[round_idx]
        wait_strat = RETRY_WAIT_STRATEGY[round_idx]

        print(f"\n   🔁 RETRY ROUND {round_num}/{MAX_RETRY_ROUNDS} | NA: {na_rate:.1%} ({na_cells}/{total_cells}) | {len(na_items)} cells", flush=True)
        print(f"   ⚡ Strategy: {wait_strat} | timeout: {timeout/1000:.0f}s", flush=True)

        cooldown = random.uniform(*RETRY_COOL_DOWN)
        print(f"   ⏳ Cooldown {cooldown:.0f}s...", flush=True)
        await asyncio.sleep(cooldown)

        hotel_na = {}
        for key, week_num in na_items:
            if key not in hotel_na:
                hotel_na[key] = []
            hotel_na[key].append(week_num)

        hotel_semaphore = asyncio.Semaphore(NUM_WORKERS)

        async def retry_hotel_weeks(key, weeks):
            async with hotel_semaphore:
                hotel_info = key_to_info[key]
                hotel_name, hotel_url, room_type = hotel_info
                print(f"   🔄 Retry {hotel_name}: W{',W'.join(str(w) for w in weeks)}", flush=True)

                await asyncio.sleep(random.uniform(*HOTEL_DELAY))

                week_semaphore = asyncio.Semaphore(WEEKS_PER_HOTEL)

                async def retry_one_week(wn):
                    async with week_semaphore:
                        offset = week_offsets[wn - 1]
                        checkin = base_checkin + timedelta(days=offset)
                        return await crawl_single_week(
                            browser_mgr, hotel_url, room_type, wn, checkin,
                            retries=MAX_RETRIES,
                            page_timeout=timeout,
                            wait_until=wait_strat
                        )

                tasks = [retry_one_week(wn) for wn in weeks]
                results = await asyncio.gather(*tasks)

                updated = 0
                for r in results:
                    if r["price"] != "NA":
                        all_week_prices[key][f"Price W{r['week']}"] = r["price"]
                        updated += 1

                if updated > 0:
                    print(f"   ✅ {hotel_name}: fixed {updated}/{len(weeks)} NA weeks", flush=True)
                else:
                    print(f"   ❌ {hotel_name}: still {len(weeks)} NA", flush=True)

        retry_tasks = [retry_hotel_weeks(key, weeks) for key, weeks in hotel_na.items()]
        await asyncio.gather(*retry_tasks)

    na_rate, na_cells, total_cells = calc_batch_na_rate(all_week_prices, batch_keys)
    if na_rate > TARGET_NA_RATE:
        print(f"   ⚠️ Batch vẫn còn NA: {na_rate:.1%} ({na_cells}/{total_cells}) sau {MAX_RETRY_ROUNDS} rounds", flush=True)
    else:
        print(f"   🎯 Batch NA rate: {na_rate:.1%} → đạt target!", flush=True)

# ============================================================
# 🚀 MAIN - BATCH + RETRY
# ============================================================
async def main():
    start_time = time.time()

    df_hotels = read_hotels_from_csv(INPUT_FILE)
    if len(df_hotels) == 0:
        print("❌ Không có hotels nào!")
        return

    all_week_prices = {}
    prev_data = {}

    # Load previous data từ temp hoặc final file (để re-run chỉ crawl NA)
    def load_prev_csv(filepath):
        """Load CSV và convert NaN → 'NA' string đúng cách"""
        loaded = 0
        try:
            df_prev = pd.read_csv(filepath, keep_default_na=False, na_values=[])
            for _, row in df_prev.iterrows():
                key = (row["hotel_name"], row["room_type"])
                if key in prev_data:
                    # Merge: chỉ ghi đè nếu giá trị mới không phải NA
                    for i in range(1, 7):
                        val = str(row.get(f"price_w{i}", "NA")).strip()
                        if val and val != "NA" and val != "nan":
                            prev_data[key][f"Price W{i}"] = val
                else:
                    p = {}
                    for i in range(1, 7):
                        val = str(row.get(f"price_w{i}", "NA")).strip()
                        p[f"Price W{i}"] = "NA" if (not val or val == "nan" or val == "NA") else val
                    prev_data[key] = p
                    all_week_prices[key] = p
                loaded += 1
        except Exception as e:
            print(f"⚠️  Lỗi đọc {filepath}: {e}", flush=True)
        return loaded

    # Load temp file trước
    if os.path.exists(TEMP_OUTPUT_FILE):
        n = load_prev_csv(TEMP_OUTPUT_FILE)
        print(f"📂 Loaded {n} hotels từ temp: {TEMP_OUTPUT_FILE}", flush=True)

    # Load final file (merge thêm, không ghi đè giá đã có)
    today_final = f"{OUTPUT_PREFIX}{datetime.today().strftime('%Y%m%d')}.csv"
    for final_file in [today_final]:
        if os.path.exists(final_file) and final_file != TEMP_OUTPUT_FILE:
            n = load_prev_csv(final_file)
            print(f"📂 Merged {n} hotels từ final: {final_file}", flush=True)

    if prev_data:
        total_prev = sum(1 for p in prev_data.values() for i in range(1,7) if p.get(f"Price W{i}","NA") != "NA")
        total_na = sum(1 for p in prev_data.values() for i in range(1,7) if p.get(f"Price W{i}","NA") == "NA")
        print(f"✅ Prev data: {len(prev_data)} hotels | {total_prev} prices | {total_na} NA to retry", flush=True)

    base_checkin = datetime.today().replace(hour=0, minute=0, second=0, microsecond=0) + timedelta(days=1)
    week_offsets = [0, 7, 14, 21, 28, 35]

    all_hotel_infos = []
    for _, row in df_hotels.iterrows():
        all_hotel_infos.append((row['hotel_name'], row['hotel_url'], row['room_type']))

    total_hotels = len(all_hotel_infos)

    print(f"\n{'='*60}", flush=True)
    print(f"🎭 BATCH CRAWL v3 - SOLD OUT + DUAL FORMAT + ESCALATION", flush=True)
    print(f"📊 Total: {total_hotels} hotels | Batch size: {BATCH_SIZE}", flush=True)
    print(f"🔁 Retry: {MAX_RETRY_ROUNDS} rounds | Target NA: ≤{TARGET_NA_RATE:.0%}", flush=True)
    print(f"⚡ {NUM_WORKERS} workers × {WEEKS_PER_HOTEL} weeks parallel", flush=True)
    print(f"🔧 Retry escalation: {RETRY_WAIT_STRATEGY}", flush=True)
    print(f"{'='*60}\n", flush=True)

    async with async_playwright() as p:
        browser_mgr = BrowserManager(p)
        await browser_mgr.get_browser()

        total_batches = (total_hotels + BATCH_SIZE - 1) // BATCH_SIZE
        crawled_total = 0

        for batch_idx in range(total_batches):
            batch_start = batch_idx * BATCH_SIZE
            batch_end = min(batch_start + BATCH_SIZE, total_hotels)
            batch_infos = all_hotel_infos[batch_start:batch_end]

            print(f"\n{'='*60}", flush=True)
            print(f"📦 BATCH {batch_idx + 1}/{total_batches} | Hotels {batch_start + 1}-{batch_end}/{total_hotels}", flush=True)
            print(f"{'='*60}", flush=True)

            hotel_semaphore = asyncio.Semaphore(NUM_WORKERS)
            tasks = [
                process_hotel(browser_mgr, info, prev_data, base_checkin, week_offsets, hotel_semaphore)
                for info in batch_infos
            ]

            batch_keys = []
            for coro in asyncio.as_completed(tasks):
                try:
                    key, prices, skipped = await coro
                    all_week_prices[key] = prices
                    batch_keys.append(key)

                    if not skipped:
                        crawled_total += 1

                    save_backup_csv(all_week_prices, TEMP_OUTPUT_FILE)
                except Exception as e:
                    print(f"❌ Error: {e}", flush=True)

            na_rate, na_cells, total_cells = calc_batch_na_rate(all_week_prices, batch_keys)
            print(f"\n📊 Batch {batch_idx + 1} initial: NA = {na_rate:.1%} ({na_cells}/{total_cells})", flush=True)

            if na_rate > TARGET_NA_RATE:
                print(f"🔁 NA > {TARGET_NA_RATE:.0%} → Retry với escalation...", flush=True)
                await retry_na_for_batch(browser_mgr, batch_infos, batch_keys, all_week_prices, base_checkin, week_offsets)
                save_backup_csv(all_week_prices, TEMP_OUTPUT_FILE)

            # Final batch report
            na_rate, na_cells, total_cells = calc_batch_na_rate(all_week_prices, batch_keys)
            print(f"\n{'─'*50}", flush=True)
            print(f"📊 BATCH {batch_idx + 1} FINAL: NA = {na_rate:.1%} ({na_cells}/{total_cells})", flush=True)
            for key in batch_keys:
                hotel_name = key[0]
                prices = all_week_prices[key]
                status_parts = []
                for i in range(1, 7):
                    val = prices.get(f"Price W{i}", "NA")
                    if val == "NA":
                        status_parts.append(f"W{i}:✗")
                    elif str(val).startswith("SOLD OUT"):
                        status_parts.append(f"W{i}:🚫")
                    else:
                        status_parts.append(f"W{i}:✓")
                status = " ".join(status_parts)
                na_c, _ = calc_na_stats(prices)
                sold_c = sum(1 for i in range(1, 7) if str(prices.get(f"Price W{i}", "")).startswith("SOLD OUT"))
                icon = "✅" if na_c == 0 else "⚠️"
                extra = f" | {sold_c} sold out" if sold_c > 0 else ""
                print(f"   {icon} {hotel_name}: {status}{extra}", flush=True)
            print(f"{'─'*50}", flush=True)

            elapsed = time.time() - start_time
            print(f"⏱️  Elapsed: {int(elapsed//60)}m{int(elapsed%60)}s | Done: {crawled_total}/{total_hotels}", flush=True)

        await browser_mgr.close()

    final_filename = f"{OUTPUT_PREFIX}{datetime.today().strftime('%Y%m%d')}.csv"
    save_backup_csv(all_week_prices, final_filename)

    # Overall summary
    total_time = time.time() - start_time
    total_cells = len(all_week_prices) * 6
    na_total = sum(
        1 for prices in all_week_prices.values()
        for i in range(1, 7) if prices.get(f"Price W{i}", "NA") == "NA"
    )
    sold_total = sum(
        1 for prices in all_week_prices.values()
        for i in range(1, 7) if str(prices.get(f"Price W{i}", "")).startswith("SOLD OUT")
    )
    price_total = total_cells - na_total - sold_total
    overall_na_rate = na_total / max(total_cells, 1)

    print(f"\n{'='*60}", flush=True)
    print(f"✅ HOÀN THÀNH!", flush=True)
    print(f"📁 Saved: {final_filename}", flush=True)
    print(f"📊 {len(all_week_prices)} hotels × 6 weeks = {total_cells} cells", flush=True)
    print(f"   ✅ Price:    {price_total} ({price_total/total_cells:.1%})", flush=True)
    print(f"   🚫 Sold out: {sold_total} ({sold_total/total_cells:.1%})", flush=True)
    print(f"   ❌ NA:       {na_total} ({overall_na_rate:.1%})", flush=True)
    if overall_na_rate <= TARGET_NA_RATE:
        print(f"🎯 ĐẠT TARGET NA ≤ {TARGET_NA_RATE:.0%}!", flush=True)
    else:
        print(f"⚠️ Chưa đạt target {TARGET_NA_RATE:.0%}", flush=True)
    print(f"⏱️  {int(total_time//60)}m {int(total_time%60)}s", flush=True)
    if crawled_total > 0:
        print(f"⚡ {total_time/crawled_total:.1f}s/hotel", flush=True)
    print(f"{'='*60}", flush=True)

# Run
await main()

⚠️  CSV columns không chuẩn. Dùng 3 columns đầu.
✅ Đọc được 229 hotels từ ./raw.csv
📂 Loaded 60 hotels từ temp: hotel_prices_temp.csv
✅ Prev data: 60 hotels | 283 prices | 77 NA to retry

🎭 BATCH CRAWL v3 - SOLD OUT + DUAL FORMAT + ESCALATION
📊 Total: 229 hotels | Batch size: 10
🔁 Retry: 2 rounds | Target NA: ≤10%
⚡ 4 workers × 3 weeks parallel
🔧 Retry escalation: ['domcontentloaded', 'networkidle']

✅ Browser launched

📦 BATCH 1/23 | Hotels 1-10/229
⏭️  SKIP: Becamex Hotel New City
⏭️  SKIP: Sheraton Can Tho
⏭️  SKIP: Citadines Central Binh Duong
⏭️  SKIP: Fairfield by Marriott South Binh Duong
⏭️  SKIP: HIIVE by fusion Binh Duong - VSIP 1
⏭️  SKIP: Legacy Mekong
⏭️  SKIP: Ana Mandara Villas Dalat Resort & Spa

🏨 START: Bach Place Dalat | Room: Deluxe Twin - Breakfast
   ✓ W1: ₫1,615,257 (cached)
   ✓ W2: ₫1,414,955 (cached)
   ✓ W4: ₫1,330,738 (cached)
   ✓ W5: ₫1,411,579 (cached)
   ✓ W6: ₫1,404,319 (cached)
   🚀 Crawling 1 weeks (max 3 parallel)...

🏨 START: Banla Boutique Hotel | 